In [ ]:
# ============================================================
# COLAB <-> HUGGING FACE MODEL MANAGER
# ============================================================

!pip install -q -U huggingface_hub

from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, snapshot_download, login, whoami

print("=" * 60)
print("        HUGGING FACE MODEL MANAGER - COLAB")
print("=" * 60)

# ------------------------------------------------------------
# LOGIN
# ------------------------------------------------------------

try:
    username = whoami()["name"]
    print(f"\n✅ Logged in as: {username}")

except Exception:
    print("\n🔐 Hugging Face login required.")
    login()
    username = whoami()["name"]
    print(f"✅ Logged in as: {username}")


# ------------------------------------------------------------
# MAIN MENU
# ------------------------------------------------------------

print("\nWhat do you want to do?")
print("1. Upload model to Hugging Face")
print("2. Download model from Hugging Face")
print("3. Exit")

choice = input("\nChoose [1/2/3]: ").strip()


# ============================================================
# UPLOAD
# ============================================================

if choice == "1":

    print("\n========== UPLOAD MODEL ==========")

    source = input(
        "\nEnter Colab FILE or FOLDER path:\n"
        "Example: /content/output/llm pretrained models/model.pth\n"
        "> "
    ).strip()

    source = Path(source)

    if not source.exists():
        print(f"\n❌ Path does not exist: {source}")
    else:

        repo_name = input(
            "\nEnter Hugging Face repo name:\n"
            "Example: my-llm-v2\n"
            "> "
        ).strip()

        repo_id = f"{username}/{repo_name}"

        private = input(
            "\nMake repository private? [Y/n]: "
        ).strip().lower() != "n"

        api = HfApi()

        try:

            # Create repo if it doesn't exist
            api.create_repo(
                repo_id=repo_id,
                repo_type="model",
                private=private,
                exist_ok=True
            )

            print(f"\n✅ Repository ready:")
            print(f"https://huggingface.co/{repo_id}")

            # ------------------------------------------------
            # FILE
            # ------------------------------------------------

            if source.is_file():

                remote_name = input(
                    f"\nFilename in HF repo [{source.name}]: "
                ).strip()

                if not remote_name:
                    remote_name = source.name

                print("\n⬆️ Uploading model...")

                api.upload_file(
                    path_or_fileobj=str(source),
                    path_in_repo=remote_name,
                    repo_id=repo_id,
                    repo_type="model"
                )

            # ------------------------------------------------
            # FOLDER
            # ------------------------------------------------

            else:

                print("\n⬆️ Uploading entire folder...")

                api.upload_folder(
                    folder_path=str(source),
                    repo_id=repo_id,
                    repo_type="model"
                )

            print("\n✅ UPLOAD COMPLETE!")
            print(f"Repository: https://huggingface.co/{repo_id}")

        except Exception as e:
            print("\n❌ Upload failed:")
            print(e)


# ============================================================
# DOWNLOAD
# ============================================================

elif choice == "2":

    print("\n========== DOWNLOAD MODEL ==========")

    repo_id = input(
        "\nEnter Hugging Face repo ID:\n"
        "Example: harsh-prajapati54/my-llm-v2\n"
        "> "
    ).strip()

    destination = input(
        "\nEnter Colab destination folder:\n"
        "Example: /content/downloaded_model\n"
        "> "
    ).strip()

    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)

    print("\nDownload:")
    print("1. One file")
    print("2. Entire repository")

    download_choice = input("\nChoose [1/2]: ").strip()

    try:

        # ------------------------------------------------
        # SINGLE FILE
        # ------------------------------------------------

        if download_choice == "1":

            filename = input(
                "\nEnter filename:\n"
                "Example: model.pth\n"
                "> "
            ).strip()

            print("\n⬇️ Downloading...")

            path = hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                repo_type="model",
                local_dir=str(destination)
            )

            print("\n✅ DOWNLOAD COMPLETE")
            print(f"Saved to: {path}")

        # ------------------------------------------------
        # ENTIRE REPOSITORY
        # ------------------------------------------------

        elif download_choice == "2":

            print("\n⬇️ Downloading repository...")

            path = snapshot_download(
                repo_id=repo_id,
                repo_type="model",
                local_dir=str(destination)
            )

            print("\n✅ DOWNLOAD COMPLETE")
            print(f"Saved to: {path}")

        else:
            print("\n❌ Invalid choice.")

    except Exception as e:
        print("\n❌ Download failed:")
        print(e)


# ============================================================
# EXIT
# ============================================================

elif choice == "3":

    print("\nBye! 👋")

else:

    print("\n❌ Invalid choice.")